# ペットボトル抽出データセット作成 Colab

目的:

- Google Drive に出力先を作成
- COCO / LVIS / TACO / Open Images などから「ボトル/ペットボトル候補」が写っているサンプルだけを取得
- 画像と COCO instance segmentation 形式のアノテーションを出力
- RTMDet-Ins / MMDetection の学習に渡せる形へ整理

注意:

- COCO や Open Images の `bottle` は「ペットボトル」だけでなく、ガラス瓶・水筒・酒瓶なども混ざります。
- TACO は `plastic bottle` 系を抽出しやすいですが、ゴミ・屋外寄りです。
- LVIS はカテゴリが細かい一方、画像は COCO 由来なので画像 URL 取得に失敗する場合があります。
- ライセンス・利用条件は各データセットの公式ページを確認してください。


In [ ]:
#@title 1. Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 2. 設定

# 出力先。必要に応じて変更してください。
OUTPUT_DIR = "/content/drive/MyDrive/ml/セグメンテーション/datasets/pet_bottle"

# 作業用キャッシュ。Colab VM側に置きます。
WORK_DIR = "/content/work_pet_bottle"

# 取得対象
USE_COCO = True
USE_LVIS = True
USE_TACO = True
USE_OPEN_IMAGES = False  # Open Imagesは重いので、まずFalse推奨

# 各データソースから最大何枚取得するか。
# None にすると上限なし。最初は 100〜500 程度で試すのがおすすめです。
MAX_SAMPLES_PER_SOURCE = None

# train/val/test 分割
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
RANDOM_SEED = 42

# COCO: 'bottle' はペットボトル以外も含みます。
COCO_SPLITS = ["train2017", "val2017"]

# LVIS: カテゴリ名に以下キーワードを含むものを抽出。
LVIS_SPLITS = ["train", "val"]
LVIS_INCLUDE_KEYWORDS = [
    "bottle", "water_bottle", "plastic_bottle", "soda_bottle"
]
LVIS_EXCLUDE_KEYWORDS = [
    "baby_bottle", "wine_bottle", "beer_bottle", "bottle_opener", "bottle_cap"
]

# TACO: plastic bottle系だけに寄せる。
TACO_INCLUDE_KEYWORDS = ["plastic bottle", "bottle"]
TACO_EXCLUDE_KEYWORDS = ["bottle cap", "glass bottle", "jar"]

# Open Images: 環境によって利用可能なクラス名が変わる可能性があるため、まず Bottle から。
OPEN_IMAGES_SPLITS = ["validation"]  # trainは大きい。最初はvalidation推奨。
OPEN_IMAGES_CLASSES = ["Bottle"]
OPEN_IMAGES_LABEL_TYPES = ["segmentations", "detections"]

# マスク/アノテーションの最小面積。小さすぎる断片を除外。
MIN_ANNOTATION_AREA = 64

# 出力カテゴリ名。RTMDet-Insではまず1クラス化がおすすめ。
UNIFIED_CATEGORY = {"id": 1, "name": "plastic_bottle", "supercategory": "bottle"}

print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
import sys
# Uninstall all existing Pillow versions to clear dependency conflicts
!{sys.executable} -m pip uninstall -y Pillow

# Install Pillow==12.2.0 to satisfy fiftyone's explicit requirement (>=12.2) and resolve _Ink import error.
# This might conflict with pre-installed gradio (requires Pillow<12.0).
!{sys.executable} -m pip install -q 'Pillow==12.2.0'

# Install fiftyone and fiftyone-db
if USE_OPEN_IMAGES:
    !{sys.executable} -m pip install -q fiftyone fiftyone-db

# Install other required packages
!{sys.executable} -m pip install -q pycocotools tqdm requests pandas scikit-learn

In [ ]:

#@title 4. 共通ユーティリティ

import os
import re
import json
import time
import math
import shutil
import zipfile
import random
import hashlib
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from urllib.parse import urlparse

import requests
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

OUT_IMAGES_DIR = Path(OUTPUT_DIR) / "images" / "all"
OUT_ANN_DIR = Path(OUTPUT_DIR) / "annotations"
OUT_META_DIR = Path(OUTPUT_DIR) / "metadata"
OUT_PREVIEW_DIR = Path(OUTPUT_DIR) / "previews"
for p in [OUT_IMAGES_DIR, OUT_ANN_DIR, OUT_META_DIR, OUT_PREVIEW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", str(s))

def download_file(url: str, dst: Path, timeout=60, retries=3, chunk_size=1024*1024):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return dst

    last_err = None
    for i in range(retries):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))
                with open(dst, "wb") as f, tqdm(
                    total=total if total else None,
                    unit="B",
                    unit_scale=True,
                    desc=dst.name,
                    leave=False,
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            return dst
        except Exception as e:
            last_err = e
            time.sleep(2 * (i + 1))
    raise RuntimeError(f"download failed: {url} -> {dst}: {last_err})")

def unzip_if_needed(zip_path: Path, dst_dir: Path):
    dst_dir.mkdir(parents=True, exist_ok=True)
    marker = dst_dir / ".unzipped"
    if marker.exists():
        return
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dst_dir)
    marker.write_text("ok", encoding="utf-8")

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def text_matches(name: str, include_keywords: List[str], exclude_keywords: Optional[List[str]] = None) -> bool:
    n = name.lower().replace("-", "_").replace(" ", "_")
    inc = [k.lower().replace("-", "_").replace(" ", "_") for k in include_keywords]
    exc = [k.lower().replace("-", "_").replace(" ", "_") for k in (exclude_keywords or [])]
    return any(k in n for k in inc) and not any(k in n for k in exc)

def get_image_url(img: dict) -> Optional[str]:
    # COCO/LVISでよくあるフィールドを順に見る
    for key in ["coco_url", "flickr_url", "url"]:
        v = img.get(key)
        if v:
            return v
    return None

def image_ext_from_url_or_filename(url: Optional[str], file_name: str) -> str:
    for s in [file_name, url or ""]:
        ext = os.path.splitext(urlparse(s).path)[1].lower()
        if ext in [".jpg", ".jpeg", ".png", ".webp"]:
            return ".jpg" if ext == ".jpeg" else ext
    return ".jpg"

def copy_or_download_image(img: dict, source: str, src_local_root: Optional[Path] = None) -> Optional[str]:
    """
    出力先 images/all に画像を保存し、COCO JSON用の相対パス file_name を返す。
    """
    img_id = img["id"]
    src_file_name = img.get("file_name", f"{img_id}.jpg")
    url = get_image_url(img)
    ext = image_ext_from_url_or_filename(url, src_file_name)
    out_name = f"{source}_{safe_name(img_id)}{ext}"
    out_path = OUT_IMAGES_DIR / out_name

    if out_path.exists() and out_path.stat().st_size > 0:
        return f"images/all/{out_name}"

    # local copy
    if src_local_root is not None:
        candidates = [
            src_local_root / src_file_name,
            src_local_root / "data" / src_file_name,
            src_local_root / "images" / src_file_name,
        ]
        for c in candidates:
            if c.exists():
                shutil.copy2(c, out_path)
                return f"images/all/{out_name}"

    # download
    if url:
        try:
            download_file(url, out_path, timeout=30, retries=3)
            return f"images/all/{out_name}"
        except Exception as e:
            print(f"[WARN] image download failed: {source} image_id={img_id} url={url} err={e}")
            return None

    print(f"[WARN] no image source: {source} image_id={img_id}")
    return None

def build_filtered_coco(
    data: dict,
    source: str,
    include_keywords: List[str],
    exclude_keywords: Optional[List[str]] = None,
    src_local_root: Optional[Path] = None,
    max_samples: Optional[int] = None,
    category_id_whitelist: Optional[List[int]] = None,
) -> Tuple[dict, pd.DataFrame]:
    """
    COCO/LVIS/TACO風JSONから対象カテゴリだけ抽出し、画像をOUTPUT_DIRへ保存し、
    カテゴリIDを1つに統一したCOCO JSONを返す。
    """
    categories = data.get("categories", [])
    if category_id_whitelist is None:
        target_cat_ids = [
            c["id"] for c in categories
            if text_matches(c.get("name", ""), include_keywords, exclude_keywords)
        ]
    else:
        target_cat_ids = list(category_id_whitelist)

    print(f"[{source}] target category ids:", target_cat_ids)
    print(f"[{source}] target categories:", [c.get("name") for c in categories if c.get("id") in target_cat_ids])

    anns = [
        a for a in data.get("annotations", [])
        if a.get("category_id") in target_cat_ids and a.get("area", 0) >= MIN_ANNOTATION_AREA
    ]

    anns_by_img = {}
    for a in anns:
        anns_by_img.setdefault(a["image_id"], []).append(a)

    image_ids = list(anns_by_img.keys())
    random.Random(RANDOM_SEED).shuffle(image_ids)
    if max_samples is not None:
        image_ids = image_ids[:max_samples]

    img_by_id = {img["id"]: img for img in data.get("images", [])}

    out_images = []
    out_anns = []
    manifest_rows = []
    next_ann_id = 1

    for img_id in tqdm(image_ids, desc=f"{source}: copy/download/filter"):
        img = img_by_id.get(img_id)
        if not img:
            continue

        rel_file = copy_or_download_image(img, source=source, src_local_root=src_local_root)
        if rel_file is None:
            continue

        # width/heightが欠ける場合は画像から取得
        width = img.get("width")
        height = img.get("height")
        if not width or not height:
            try:
                with Image.open(Path(OUTPUT_DIR) / rel_file) as im:
                    width, height = im.size
            except Exception:
                width, height = 0, 0

        new_img_id = len(out_images) + 1
        out_images.append({
            "id": new_img_id,
            "file_name": rel_file,
            "width": width,
            "height": height,
            "source": source,
            "source_image_id": img_id,
        })

        kept = 0
        for a in anns_by_img[img_id]:
            na = dict(a)
            na["id"] = next_ann_id
            na["image_id"] = new_img_id
            na["category_id"] = UNIFIED_CATEGORY["id"]
            na["source"] = source
            na["source_annotation_id"] = a.get("id")
            out_anns.append(na)
            next_ann_id += 1
            kept += 1

        manifest_rows.append({
            "source": source,
            "source_image_id": img_id,
            "file_name": rel_file,
            "num_annotations": kept,
            "original_file_name": img.get("file_name"),
            "url": get_image_url(img),
        })

    out = {
        "info": {
            "description": "Pet bottle candidate instance segmentation dataset extracted from public datasets",
            "source": source,
        },
        "licenses": data.get("licenses", []),
        "images": out_images,
        "annotations": out_anns,
        "categories": [UNIFIED_CATEGORY],
    }
    return out, pd.DataFrame(manifest_rows)

def merge_coco_jsons(coco_jsons: List[dict]) -> dict:
    merged = {
        "info": {"description": "Merged pet bottle candidate dataset"},
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [UNIFIED_CATEGORY],
    }
    next_img_id = 1
    next_ann_id = 1

    for ds in coco_jsons:
        id_map = {}
        for img in ds.get("images", []):
            new_img = dict(img)
            old_id = new_img["id"]
            new_img["id"] = next_img_id
            id_map[old_id] = next_img_id
            merged["images"].append(new_img)
            next_img_id += 1

        for ann in ds.get("annotations", []):
            if ann["image_id"] not in id_map:
                continue
            new_ann = dict(ann)
            new_ann["id"] = next_ann_id
            new_ann["image_id"] = id_map[ann["image_id"]]
            new_ann["category_id"] = UNIFIED_CATEGORY["id"]
            merged["annotations"].append(new_ann)
            next_ann_id += 1

    return merged

def subset_coco_by_image_ids(coco: dict, image_ids: List[int]) -> dict:
    image_ids = set(image_ids)
    anns = [a for a in coco["annotations"] if a["image_id"] in image_ids]
    imgs = [i for i in coco["images"] if i["id"] in image_ids]
    return {
        "info": dict(coco.get("info", {})),
        "licenses": coco.get("licenses", []),
        "images": imgs,
        "annotations": anns,
        "categories": coco["categories"],
    }

def write_train_val_test(coco: dict):
    image_ids = [img["id"] for img in coco["images"]]
    if len(image_ids) < 3:
        save_json(coco, OUT_ANN_DIR / "instances_all.json")
        return

    train_ids, temp_ids = train_test_split(
        image_ids,
        train_size=TRAIN_RATIO,
        random_state=RANDOM_SEED,
        shuffle=True,
    )
    val_fraction_in_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    val_ids, test_ids = train_test_split(
        temp_ids,
        train_size=val_fraction_in_temp,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    save_json(coco, OUT_ANN_DIR / "instances_all.json")
    save_json(subset_coco_by_image_ids(coco, train_ids), OUT_ANN_DIR / "instances_train.json")
    save_json(subset_coco_by_image_ids(coco, val_ids), OUT_ANN_DIR / "instances_val.json")
    save_json(subset_coco_by_image_ids(coco, test_ids), OUT_ANN_DIR / "instances_test.json")

    print("saved:")
    print(" all :", OUT_ANN_DIR / "instances_all.json", len(coco["images"]), "images")
    print(" train:", len(train_ids), "images")
    print(" val :", len(val_ids), "images")
    print(" test:", len(test_ids), "images")

def coco_stats(coco: dict) -> pd.DataFrame:
    return pd.DataFrame([{
        "images": len(coco.get("images", [])),
        "annotations": len(coco.get("annotations", [])),
        "categories": [c["name"] for c in coco.get("categories", [])],
    }])

In [ ]:

#@title 5. COCO 2017から bottle を抽出

source_cocos = []
manifests = []

if USE_COCO:
    coco_ann_zip_url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
    coco_dir = Path(WORK_DIR) / "coco"
    zip_path = download_file(coco_ann_zip_url, coco_dir / "annotations_trainval2017.zip")
    unzip_if_needed(zip_path, coco_dir)

    for split in COCO_SPLITS:
        ann_path = coco_dir / "annotations" / f"instances_{split}.json"
        data = load_json(ann_path)

        # COCOはカテゴリ名 bottle を直接指定
        bottle_ids = [c["id"] for c in data["categories"] if c["name"] == "bottle"]
        filtered, manifest = build_filtered_coco(
            data=data,
            source=f"coco_{split}",
            include_keywords=["bottle"],
            category_id_whitelist=bottle_ids,
            max_samples=MAX_SAMPLES_PER_SOURCE,
        )
        save_json(filtered, OUT_ANN_DIR / f"instances_coco_{split}.json")
        manifest.to_csv(OUT_META_DIR / f"manifest_coco_{split}.csv", index=False)
        source_cocos.append(filtered)
        manifests.append(manifest)

        display(coco_stats(filtered))
else:
    print("USE_COCO=False")


In [ ]:

#@title 6. LVIS v1から bottle系カテゴリを抽出

if USE_LVIS:
    # 公式で使われる公開URL。失敗する場合はLVIS公式ページからJSONを手動DLして WORK_DIR/lvis に置いてください。
    lvis_urls = {
        "train": "https://dl.fbaipublicfiles.com/LVIS/lvis_v1_train.json.zip",
        "val": "https://dl.fbaipublicfiles.com/LVIS/lvis_v1_val.json.zip",
    }
    lvis_dir = Path(WORK_DIR) / "lvis"
    lvis_dir.mkdir(parents=True, exist_ok=True)

    for split in LVIS_SPLITS:
        try:
            zip_path = download_file(lvis_urls[split], lvis_dir / f"lvis_v1_{split}.json.zip")
            unzip_if_needed(zip_path, lvis_dir / split)
            # zip内の名前が lvis_v1_train.json / lvis_v1_val.json である前提
            ann_candidates = list((lvis_dir / split).glob("*.json"))
            if not ann_candidates:
                raise FileNotFoundError("LVIS json not found after unzip")
            ann_path = ann_candidates[0]
        except Exception as e:
            manual_path = lvis_dir / f"lvis_v1_{split}.json"
            if manual_path.exists():
                ann_path = manual_path
            else:
                print(f"[WARN] LVIS {split} download/load skipped: {e}")
                continue

        data = load_json(ann_path)
        filtered, manifest = build_filtered_coco(
            data=data,
            source=f"lvis_{split}",
            include_keywords=LVIS_INCLUDE_KEYWORDS,
            exclude_keywords=LVIS_EXCLUDE_KEYWORDS,
            max_samples=MAX_SAMPLES_PER_SOURCE,
        )
        save_json(filtered, OUT_ANN_DIR / f"instances_lvis_{split}.json")
        manifest.to_csv(OUT_META_DIR / f"manifest_lvis_{split}.csv", index=False)
        source_cocos.append(filtered)
        manifests.append(manifest)

        display(coco_stats(filtered))
else:
    print("USE_LVIS=False")


In [ ]:

#@title 7. TACOから plastic bottle / bottle 系を抽出

if USE_TACO:
    taco_dir = Path(WORK_DIR) / "TACO"
    if not taco_dir.exists():
        !git clone --depth 1 https://github.com/pedropro/TACO.git {str(taco_dir)}

    # 画像が未取得ならTACO公式スクリプトで取得
    # Flickr由来のため、URL切れが一部発生する可能性があります。
    try:
        subprocess.run(["python3", "download.py"], cwd=str(taco_dir), check=False)
    except Exception as e:
        print("[WARN] TACO download.py failed:", e)

    ann_path = taco_dir / "data" / "annotations.json"
    if not ann_path.exists():
        raise FileNotFoundError(f"TACO annotation not found: {ann_path}")

    data = load_json(ann_path)
    filtered, manifest = build_filtered_coco(
        data=data,
        source="taco",
        include_keywords=TACO_INCLUDE_KEYWORDS,
        exclude_keywords=TACO_EXCLUDE_KEYWORDS,
        src_local_root=taco_dir,
        max_samples=MAX_SAMPLES_PER_SOURCE,
    )
    save_json(filtered, OUT_ANN_DIR / "instances_taco.json")
    manifest.to_csv(OUT_META_DIR / "manifest_taco.csv", index=False)
    source_cocos.append(filtered)
    manifests.append(manifest)

    display(coco_stats(filtered))
else:
    print("USE_TACO=False")


In [ ]:

#@title 8. Open Images V7から Bottle を抽出（任意・重い）

if USE_OPEN_IMAGES:
    import fiftyone as fo
    import fiftyone.zoo as foz

    oi_export_cocos = []

    for split in OPEN_IMAGES_SPLITS:
        print("Loading Open Images:", split)
        # max_samplesで強く制限することを推奨
        ds = foz.load_zoo_dataset(
            "open-images-v7",
            split=split,
            label_types=OPEN_IMAGES_LABEL_TYPES,
            classes=OPEN_IMAGES_CLASSES,
            max_samples=MAX_SAMPLES_PER_SOURCE,
            only_matching=True,
            shuffle=True,
            dataset_name=f"open_images_pet_bottle_{split}_{int(time.time())}",
        )

        # segmentationsがあればそれを優先。なければdetectionsを使うが、RTMDet-Ins学習にはマスクが不足する可能性あり。
        label_field = None
        for f in ["segmentations", "detections", "ground_truth"]:
            if f in ds.get_field_schema():
                label_field = f
                break

        if label_field is None:
            print("[WARN] Open Images label field not found")
            continue

        export_dir = Path(WORK_DIR) / f"open_images_export_{split}"
        if export_dir.exists():
            shutil.rmtree(export_dir)

        ds.export(
            export_dir=str(export_dir),
            dataset_type=fo.types.COCODetectionDataset,
            label_field=label_field,
            export_media=True,
        )

        # FiftyOneのCOCO export結果を読み込み、再度統一形式へ寄せる
        ann_candidates = list(export_dir.glob("**/*.json"))
        if not ann_candidates:
            print("[WARN] Open Images export json not found")
            continue

        data = load_json(ann_candidates[0])

        # export済み画像はローカルコピーとして処理
        filtered, manifest = build_filtered_coco(
            data=data,
            source=f"open_images_{split}",
            include_keywords=["bottle"],
            src_local_root=export_dir,
            max_samples=MAX_SAMPLES_PER_SOURCE,
        )
        save_json(filtered, OUT_ANN_DIR / f"instances_open_images_{split}.json")
        manifest.to_csv(OUT_META_DIR / f"manifest_open_images_{split}.csv", index=False)
        source_cocos.append(filtered)
        manifests.append(manifest)

        display(coco_stats(filtered))
else:
    print("USE_OPEN_IMAGES=False")


In [ ]:

#@title 9. 手動配置済みCOCO形式データも追加する場合

# RPC / HOIST / VISOR / 自前SAM3出力など、手動でDriveに置いたCOCO形式データを追加したい場合に使います。
# 例:
# MANUAL_COCO_JSONS = [
#     "/content/drive/MyDrive/manual_petbottle/annotations/instances_train.json",
# ]
# MANUAL_IMAGE_ROOTS = [
#     "/content/drive/MyDrive/manual_petbottle",
# ]
MANUAL_COCO_JSONS = []
MANUAL_IMAGE_ROOTS = []

for json_path, image_root in zip(MANUAL_COCO_JSONS, MANUAL_IMAGE_ROOTS):
    json_path = Path(json_path)
    image_root = Path(image_root)
    data = load_json(json_path)

    filtered, manifest = build_filtered_coco(
        data=data,
        source=f"manual_{safe_name(json_path.stem)}",
        include_keywords=["bottle", "plastic_bottle", "pet_bottle"],
        exclude_keywords=["bottle_cap", "cap"],
        src_local_root=image_root,
        max_samples=MAX_SAMPLES_PER_SOURCE,
    )
    save_json(filtered, OUT_ANN_DIR / f"instances_manual_{safe_name(json_path.stem)}.json")
    manifest.to_csv(OUT_META_DIR / f"manifest_manual_{safe_name(json_path.stem)}.csv", index=False)
    source_cocos.append(filtered)
    manifests.append(manifest)

    display(coco_stats(filtered))


In [ ]:

#@title 10. 全ソースをマージして train/val/test を出力

if not source_cocos:
    raise RuntimeError("抽出されたデータがありません。USE_COCO等の設定を確認してください。")

merged = merge_coco_jsons(source_cocos)
write_train_val_test(merged)

if manifests:
    manifest_all = pd.concat(manifests, ignore_index=True)
    manifest_all.to_csv(OUT_META_DIR / "manifest_all.csv", index=False)
    display(manifest_all.head())
    print("manifest:", OUT_META_DIR / "manifest_all.csv")

display(coco_stats(merged))


In [ ]:

#@title 11. プレビュー画像を作成

import matplotlib.pyplot as plt
from pycocotools import mask as maskUtils
import numpy as np
from PIL import Image, ImageDraw

def ann_to_mask(ann, h, w):
    seg = ann.get("segmentation")
    if seg is None:
        return None
    try:
        if isinstance(seg, list):
            rles = maskUtils.frPyObjects(seg, h, w)
            rle = maskUtils.merge(rles)
        elif isinstance(seg, dict) and isinstance(seg.get("counts"), list):
            rle = maskUtils.frPyObjects(seg, h, w)
        else:
            rle = seg
        m = maskUtils.decode(rle)
        if m.ndim == 3:
            m = np.any(m, axis=2)
        return m.astype(bool)
    except Exception:
        return None

def make_previews(coco, n=12):
    anns_by_img = {}
    for a in coco["annotations"]:
        anns_by_img.setdefault(a["image_id"], []).append(a)

    imgs = [img for img in coco["images"] if img["id"] in anns_by_img]
    random.Random(RANDOM_SEED).shuffle(imgs)
    imgs = imgs[:n]

    for idx, img in enumerate(imgs):
        img_path = Path(OUTPUT_DIR) / img["file_name"]
        if not img_path.exists():
            continue
        im = Image.open(img_path).convert("RGB")
        overlay = Image.new("RGBA", im.size, (0,0,0,0))
        draw = ImageDraw.Draw(overlay)

        for ann in anns_by_img.get(img["id"], []):
            bbox = ann.get("bbox")
            if bbox:
                x, y, bw, bh = bbox
                draw.rectangle([x, y, x+bw, y+bh], outline=(255,0,0,255), width=3)

            m = ann_to_mask(ann, img.get("height", im.height), img.get("width", im.width))
            if m is not None:
                mask_img = Image.fromarray((m * 120).astype(np.uint8), mode="L").resize(im.size)
                color = Image.new("RGBA", im.size, (255, 0, 0, 90))
                overlay = Image.composite(color, overlay, mask_img)

        out = Image.alpha_composite(im.convert("RGBA"), overlay).convert("RGB")
        out_path = OUT_PREVIEW_DIR / f"preview_{idx:03d}.jpg"
        out.save(out_path, quality=90)

    print("preview dir:", OUT_PREVIEW_DIR)

make_previews(merged, n=12)


In [ ]:
import sys
import os

# 1. Force reinstall Pillow 12.2.0
!{sys.executable} -m pip install -U --force-reinstall -q 'Pillow==12.2.0'

print("--- IMPORTANT ---")
print("Pillow has been reinstalled. Due to how Colab handles C-extensions, ")
print("you MUST restart the runtime to apply the changes.")
print("Click 'Runtime' -> 'Restart session' in the top menu, then run the utility and preview cells again.")

# Attempt to auto-restart (this will kill the current process to force a reload)
# os.kill(os.getpid(), 9)

In [ ]:
print("統計情報を確認する")
if 'manifest_all' in locals():
    source_stats = manifest_all.groupby('source').agg(
        image_count=('file_name', 'nunique'),
        annotation_count=('num_annotations', 'sum')
    ).reset_index()
    display(source_stats)
else:
    print("Error: 'manifest_all' DataFrame not found. Please run cell 10 to generate it.")

In [ ]:

#@title 12. MMDetection / RTMDet-Ins 用 dataset 設定スニペットを保存

snippet = f"""
# MMDetection config snippet
# 例: RTMDet-Ins config の data_root / metainfo / dataloader を以下に寄せる

data_root = '{OUTPUT_DIR}/'
metainfo = {{
    'classes': ('plastic_bottle',),
    # paletteは任意
    'palette': [(220, 20, 60)],
}}

train_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_train.json',
        data_prefix=dict(img=''),
    )
)

val_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_val.json',
        data_prefix=dict(img=''),
    )
)

test_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_test.json',
        data_prefix=dict(img=''),
    )
)

val_evaluator = dict(ann_file=data_root + 'annotations/instances_val.json')
test_evaluator = dict(ann_file=data_root + 'annotations/instances_test.json')
"""

path = Path(OUTPUT_DIR) / "mmdet_dataset_snippet.py"
path.write_text(snippet, encoding="utf-8")
print(path)
print(snippet)


In [ ]:
import os
from pathlib import Path

# 変更したい場合はここを編集してください
new_classes = ('plastic_bottle',)
new_palette = [(220, 20, 60)] # RGB

snippet = f"""
# MMDetection config snippet
data_root = '{OUTPUT_DIR}/'
metainfo = {{
    'classes': {new_classes},
    'palette': {new_palette},
}}

train_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_train.json',
        data_prefix=dict(img=''),
    )
)

val_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_val.json',
        data_prefix=dict(img=''),
    )
)

test_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_test.json',
        data_prefix=dict(img=''),
    )
)

val_evaluator = dict(ann_file=data_root + 'annotations/instances_val.json')
test_evaluator = dict(ann_file=data_root + 'annotations/instances_test.json')
"""

snippet_path = Path(OUTPUT_DIR) / "mmdet_dataset_snippet.py"
snippet_path.write_text(snippet, encoding='utf-8')

print(f"--- {snippet_path.name} を更新しました ---")
print(snippet)

In [ ]:
import os
import sys

# Force reinstall and then KILL the runtime to force a fresh start on next run
print("Pillowを強制再インストールしてランタイムを強制終了します...")
!{sys.executable} -m pip install -U --force-reinstall 'Pillow==12.2.0'

print("\n!!! ランタイムを再起動しています。終了後、Cell 4 と Cell 10 を再実行してください。 !!!")
os.kill(os.getpid(), 9)

In [ ]:
import PIL
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from pathlib import Path
import random
import numpy as np
import json
from pycocotools import mask as maskUtils

# 再起動後に失われた設定を再定義
OUTPUT_DIR = "/content/drive/MyDrive/ml/セグメンテーション/datasets/pet_bottle"
OUT_PREVIEW_DIR = Path(OUTPUT_DIR) / "previews"
RANDOM_SEED = 42

print(f"Loaded Pillow version: {PIL.__version__}")

# プレビュー作成に必要な関数を再定義 (Cell 11の内容)
def ann_to_mask(ann, h, w):
    seg = ann.get("segmentation")
    if seg is None: return None
    try:
        if isinstance(seg, list):
            rles = maskUtils.frPyObjects(seg, h, w)
            rle = maskUtils.merge(rles)
        elif isinstance(seg, dict) and isinstance(seg.get("counts"), list):
            rle = maskUtils.frPyObjects(seg, h, w)
        else:
            rle = seg
        m = maskUtils.decode(rle)
        if m.ndim == 3: m = np.any(m, axis=2)
        return m.astype(bool)
    except Exception: return None

def make_previews(coco, n=12):
    anns_by_img = {}
    for a in coco["annotations"]:
        anns_by_img.setdefault(a["image_id"], []).append(a)
    imgs = [img for img in coco["images"] if img["id" ] in anns_by_img]
    random.Random(RANDOM_SEED).shuffle(imgs)
    imgs = imgs[:n]
    OUT_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
    for idx, img in enumerate(imgs):
        img_path = Path(OUTPUT_DIR) / img["file_name"]
        if not img_path.exists(): continue
        im = Image.open(img_path).convert("RGB")
        overlay = Image.new("RGBA", im.size, (0,0,0,0))
        draw = ImageDraw.Draw(overlay)
        for ann in anns_by_img.get(img["id"], []):
            bbox = ann.get("bbox")
            if bbox:
                x, y, bw, bh = bbox
                draw.rectangle([x, y, x+bw, y+bh], outline=(255,0,0,255), width=3)
            m = ann_to_mask(ann, img.get("height", im.height), img.get("width", im.width))
            if m is not None:
                mask_img = Image.fromarray((m * 120).astype(np.uint8), mode="L").resize(im.size)
                color = Image.new("RGBA", im.size, (255, 0, 0, 90))
                overlay = Image.composite(color, overlay, mask_img)
        out = Image.alpha_composite(im.convert("RGBA"), overlay).convert("RGB")
        out_path = OUT_PREVIEW_DIR / f"preview_{idx:03d}.jpg"
        out.save(out_path, quality=90)
    print("preview dir:", OUT_PREVIEW_DIR)

# 既存の instances_all.json を読み込んでプレビューを作成
ann_path = Path(OUTPUT_DIR) / "annotations" / "instances_all.json"

if ann_path.exists():
    print(f"Loading dataset from {ann_path}...")
    with open(ann_path, 'r') as f:
        merged_data = json.load(f)
    make_previews(merged_data, n=12)
else:
    print(f"エラー: {ann_path} が見つかりません。マージ処理を再実行してください。")

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

# プレビュー画像のリストを取得
preview_files = sorted(glob.glob(str(OUT_PREVIEW_DIR / "*.jpg")))

if not preview_files:
    print("プレビュー画像が見つかりませんでした。")
else:
    # 最大4枚まで表示
    n_show = min(len(preview_files), 4)
    plt.figure(figsize=(20, 10))
    for i in range(n_show):
        plt.subplot(1, n_show, i + 1)
        img = Image.open(preview_files[i])
        plt.imshow(img)
        plt.title(Path(preview_files[i]).name)
        plt.axis('off')
    plt.show()

## 次にやること

1. `MAX_SAMPLES_PER_SOURCE = 100` 程度で実行してプレビューを確認
2. `previews/` を見て、ガラス瓶・缶・キャップなどの混入を確認
3. 混入が多い場合は `EXCLUDE_KEYWORDS` を調整
4. `annotations/instances_train.json` を RTMDet-Ins 学習ノートブックに渡す
5. 精度を上げる段階で、SAM3やCVATで誤ラベルを修正

`bottle` ラベルは「ペットボトル専用」ではないため、実運用向けには必ずQA/手直しが必要です。
